# 공공데이터포털 오픈 API 실습 - 식품안전나라 건강기능식품 영양DB 조회

> [식품안전나라 오픈API 상세 - 건강기능식품 영양DB(I0760)](https://www.foodsafetykorea.go.kr/api/openApiInfo.do?svc_no=I0760)

`.env` 파일에 저장한 인증키로 식품안전나라 오픈 API를 직접 호출해보는 실습입니다.

> 인증키 발급 방법은 [`공공데이터활용 - 인증키 생성.md`](./공공데이터활용%20-%20인증키%20생성.md) 문서를 참고하세요.

## 0. 준비하기

1. 이 폴더의 `.env.sample`을 복사해 `.env` 파일을 만든다.
2. 발급받은 인증키를 `FOOD_SAFETY_API_KEY` 값으로 입력한다.

```
FOOD_SAFETY_API_KEY=발급받은_인증키
```

> 건강기능식품 영양DB(I0760)는 `sample` 키를 지원합니다. 인증키가 없어도 실습할 수 있지만, 인증키 관리 방법을 익히려면 발급받은 키를 `.env`에 입력하세요.

In [ ]:
import os
import json

import requests
import pandas as pd
from dotenv import load_dotenv #dotenv에 있는글자를 환경변수에 저장.

load_dotenv()

API_KEY = os.getenv("FOOD_SAFETY_API_KEY")
if not API_KEY or API_KEY == "발급받은_인증키를_입력하세요":
    API_KEY = "sample"  # I0760에서 지원하는 테스트용 키

"인증키 로드 완료"  # 인증키 값은 일부라도 출력하지 않는다.

'인증키 로드 완료'

## 1. 요청 URL 구조 알아보기

API 문서에서 확인한 요청 주소 형식은 다음과 같습니다.

```
http://openapi.foodsafetykorea.go.kr/api/{인증키}/{서비스ID}/{요청타입}/{시작위치}/{종료위치}
```

| 순서 | 이름 | 설명 |
| --- | --- | --- |
| 1 | 인증키 | 발급받은 API Key (또는 테스트용 `sample`) |
| 2 | 서비스ID | 데이터셋을 구분하는 코드 (예: `I0760` = 건강기능식품 영양DB) |
| 3 | 요청타입 | 응답 형식 (`json` 또는 `xml`) |
| 4 | 시작위치 | 조회를 시작할 행 번호 (1부터 시작) |
| 5 | 종료위치 | 조회를 마칠 행 번호 (한 번에 최대 1000건) |

In [2]:
BASE_URL = "http://openapi.foodsafetykorea.go.kr/api"
SERVICE_ID = "I0760"  # 건강기능식품 영양DB
DATA_TYPE = "json"
START_IDX = 1
END_IDX = 5

url = f"{BASE_URL}/{API_KEY}/{SERVICE_ID}/{DATA_TYPE}/{START_IDX}/{END_IDX}"
safe_url = f"{BASE_URL}/<API_KEY>/{SERVICE_ID}/{DATA_TYPE}/{START_IDX}/{END_IDX}"
safe_url  # 인증키를 가린 요청 URL만 확인

'http://openapi.foodsafetykorea.go.kr/api/<API_KEY>/I0760/json/1/5'

## 2. API 호출하고 응답 확인하기

In [3]:
try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()  # HTTP 오류(4xx, 5xx)가 있으면 예외 발생
except requests.RequestException as error:
    # requests의 기본 오류 메시지에는 인증키가 담긴 URL이 표시될 수 있어 오류 유형만 알립니다.
    raise RuntimeError(f"API 요청 실패: {type(error).__name__}") from None
print("status code:", response.status_code)

data = response.json()
print(json.dumps(data, ensure_ascii=False, indent=2)[:1000])

status code: 200
{
  "I0760": {
    "total_count": "585",
    "row": [
      {
        "LCLAS_NM": "건강기능식품",
        "SCLAS_CD": "599000000",
        "HELT_ITM_GRP_NM": "프랑스해안송껍질추출물",
        "MLSFC_NM": "개별인정형 건강기능식품",
        "SCLAS_NM": "개별인정형 건강기능식품",
        "HELT_ITM_GRP_CD": "A00024",
        "LCLAS_CD": "700000000",
        "MLSFC_CD": "799000000"
      },
      {
        "LCLAS_NM": "건강기능식품",
        "SCLAS_CD": "599000000",
        "HELT_ITM_GRP_NM": "이소말토올리고당",
        "MLSFC_NM": "개별인정형 건강기능식품",
        "SCLAS_NM": "개별인정형 건강기능식품",
        "HELT_ITM_GRP_CD": "A00025",
        "LCLAS_CD": "700000000",
        "MLSFC_CD": "799000000"
      },
      {
        "LCLAS_NM": "건강기능식품",
        "SCLAS_CD": "599000000",
        "HELT_ITM_GRP_NM": "황금추출물등복합물",
        "MLSFC_NM": "개별인정형 건강기능식품",
        "SCLAS_NM": "개별인정형 건강기능식품",
        "HELT_ITM_GRP_CD": "A00026",
        "LCLAS_CD": "700000000",
        "MLSFC_CD": "799000000"
      },
      {
        "LCLAS_NM": "건강기능식품",
        

### 응답 구조 이해하기

응답은 `{서비스ID: {...}}` 형태의 딕셔너리이며, 그 안에 다음 항목이 들어 있습니다.

| 키 | 설명 |
| --- | --- |
| `total_count` | 전체 데이터 건수 |
| `row` | 실제 데이터 목록 (정상일 때만 존재) |
| `RESULT.CODE` | 처리 결과 코드 |
| `RESULT.MSG` | 처리 결과 메시지 |

In [4]:
result = data[SERVICE_ID]["RESULT"]
print(result["CODE"], "-", result["MSG"])

INFO-000 - 정상처리되었습니다.


> `INFO-000`이면 정상 응답입니다. `INFO-100`이면 `.env`의 인증키를 확인하거나 테스트용 `sample` 키를 사용하세요.

## 3. 응답을 데이터프레임으로 변환하기

In [5]:
def call_food_api(service_id, api_key=API_KEY, data_type="json", start_idx=1, end_idx=5, **params):
    """식품안전나라 오픈 API를 호출하여 (결과 코드, 데이터프레임)을 반환한다."""
    url = f"{BASE_URL}/{api_key}/{service_id}/{data_type}/{start_idx}/{end_idx}"
    if params:
        query = "&".join(f"{key}={value}" for key, value in params.items())
        url += f"/{query}"

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
    except requests.RequestException as error:
        # 인증키가 담긴 URL 대신 오류 유형만 알린다.
        raise RuntimeError(f"API 요청 실패: {type(error).__name__}") from None
    data = response.json()
    if service_id not in data:
        raise KeyError(f"응답에 서비스 ID '{service_id}'가 없습니다: {list(data)}")

    body = data[service_id]
    result = body["RESULT"]

    if result["CODE"] != "INFO-000":
        print(f"[{result['CODE']}] {result['MSG']}")
        return result["CODE"], pd.DataFrame()

    return result["CODE"], pd.DataFrame(body.get("row", []))


code_, df_food = call_food_api(SERVICE_ID, end_idx=20)
df_food.head()

,LCLAS_NM,SCLAS_CD,HELT_ITM_GRP_NM,MLSFC_NM,SCLAS_NM,HELT_ITM_GRP_CD,LCLAS_CD,MLSFC_CD
0,건강기능식품,599000000,프랑스해안송껍질추출물,개별인정형 건강기능식품,개별인정형 건강기능식품,A00024,700000000,799000000
1,건강기능식품,599000000,이소말토올리고당,개별인정형 건강기능식품,개별인정형 건강기능식품,A00025,700000000,799000000
2,건강기능식품,599000000,황금추출물등복합물,개별인정형 건강기능식품,개별인정형 건강기능식품,A00026,700000000,799000000
3,건강기능식품,599000000,사탕수수왁스알코올,개별인정형 건강기능식품,개별인정형 건강기능식품,A00027,700000000,799000000
4,건강기능식품,599000000,Dimethylsulfone(MSM),개별인정형 건강기능식품,개별인정형 건강기능식품,A00029,700000000,799000000


In [6]:
display_columns = ["HELT_ITM_GRP_CD", "HELT_ITM_GRP_NM", "LCLAS_CD", "LCLAS_NM", "MLSFC_CD", "MLSFC_NM", "SCLAS_CD", "SCLAS_NM"]
df_food.loc[:, display_columns] if not df_food.empty else df_food

,HELT_ITM_GRP_CD,HELT_ITM_GRP_NM,LCLAS_CD,LCLAS_NM,MLSFC_CD,MLSFC_NM,SCLAS_CD,SCLAS_NM
0,A00024,프랑스해안송껍질추출물,700000000,건강기능식품,799000000,개별인정형 건강기능식품,599000000,개별인정형 건강기능식품
1,A00025,이소말토올리고당,700000000,건강기능식품,799000000,개별인정형 건강기능식품,599000000,개별인정형 건강기능식품
2,A00026,황금추출물등복합물,700000000,건강기능식품,799000000,개별인정형 건강기능식품,599000000,개별인정형 건강기능식품
3,A00027,사탕수수왁스알코올,700000000,건강기능식품,799000000,개별인정형 건강기능식품,599000000,개별인정형 건강기능식품
4,A00029,Dimethylsulfone(MSM),700000000,건강기능식품,799000000,개별인정형 건강기능식품,599000000,개별인정형 건강기능식품


## 4. 조건을 추가해서 검색하기

건강기능식품 영양DB API는 `HELT_ITM_GRP_NM`(건강 항목 그룹 명)을 선택 조건으로 지원합니다.

앞서 조회한 결과에서 항목명을 하나 골라, 같은 이름의 데이터만 다시 조회해 봅니다.

In [7]:
if not df_food.empty:
    target_name = df_food.iloc[0]["HELT_ITM_GRP_NM"]
    print(f"조회할 건강 항목 그룹: {target_name}")

    code_, df_target = call_food_api(SERVICE_ID, end_idx=50, HELT_ITM_GRP_NM=target_name)
    df_target.loc[:, display_columns]

조회할 건강 항목 그룹: 프랑스해안송껍질추출물


## 5. 에러 코드 확인하기

자주 만나는 결과 코드는 다음과 같습니다.

| 코드 | 의미 |
| --- | --- |
| `INFO-000` | 정상 처리 |
| `INFO-200` | 해당 조건의 데이터 없음 |
| `INFO-100` | 인증키가 없거나 유효하지 않음 |
| `ERROR-300` | 필수 요청 파라미터 누락 |
| `ERROR-336` | 데이터 요청 범위 초과 (최대 1000건) |
| `ERROR-310` | 해당하는 서비스를 찾을 수 없음 |
| `ERROR-500` | 서버 오류 |

> 전체 오류 코드는 [식품안전나라 오픈API 가이드](https://www.foodsafetykorea.go.kr/api/howToUseApi.do?menu_grp=MENU_GRP34&menu_no=687)에서 확인할 수 있습니다.

## 6. 저장하기

조회한 결과를 `data/health_functional_food_nutrition_db.csv`로 저장해 봅니다. `utf-8-sig`는 Excel에서 한글이 깨지지 않도록 도와줍니다.

In [8]:
os.makedirs("data", exist_ok=True)
if not df_food.empty:
    df_food.to_csv("data/health_functional_food_nutrition_db.csv", index=False, encoding="utf-8-sig")
    print("저장 완료")

저장 완료
